# Procesamiento de Lenguaje Natural — Guía de Estudio
## Semanas 7 y 8: Pipeline completo Audio → Texto → Temas → LLM
### MNA · Tecnológico de Monterrey

> **¿Para qué sirve este notebook?**  
> Versión comentada a detalle de la actividad. Explica el *por qué* de cada decisión, la matemática detrás de cada modelo, y los casos de uso reales de cada etapa.

## 🗺️ Sumario — ¿Qué hace este notebook?

Construye un **pipeline de NLP de extremo a extremo** sobre 10 fábulas de Esopo en audio:

```
Audios MP3 (Gutenberg)
    │
    ▼  [Ej 1]  Descarga vía HTTP con requests
    ▼  [Ej 2a] Whisper-small → Texto transcrito
    ▼  [Ej 2b] Regex → Eliminar intro/outro comunes → JSON
    ▼  [Ej 3]  NLTK → Tokens limpios (sin stopwords)
    ▼  [Ej 4]  LDA (gensim) → 20 palabras clave por fábula
    ▼  [Ej 5]  Groq / Llama 3.3 70B → Resumen + 3 subtemas
```

---

## 🔑 Conceptos clave

| Concepto | Qué es | Dónde aparece |
|----------|--------|---------------|
| **ASR** | Automatic Speech Recognition — convertir audio a texto | Ej 2a |
| **Transformer** | Arquitectura neuronal basada en mecanismo de atención | Whisper, Llama |
| **Mel-spectrogram** | Representación visual del audio que Whisper usa como entrada | Interno en Whisper |
| **LDA** | Latent Dirichlet Allocation — modelo probabilístico de temas | Ej 4 |
| **Stopwords** | Palabras vacías sin significado temático ("de", "la", "que") | Ej 3 |
| **Tokenización** | Separar texto en unidades mínimas (palabras) | Ej 3 |
| **Prompt engineering** | Diseñar la instrucción al LLM para obtener formato específico | Ej 5 |
| **LPU** | Language Processing Unit — chip de Groq para inferencia rápida | Ej 5 |

---

## 💼 Casos de uso de negocio

| Industria | Problema real que resuelve este pipeline |
|-----------|------------------------------------------|
| **Medios / Streaming** | Subtítulos automáticos y resúmenes de episodios de podcast |
| **Legal / Compliance** | Transcribir audiencias, extraer temas, resumir para abogados |
| **Educación** | Convertir clases grabadas en notas estructuradas con temas |
| **Call centers** | Transcribir llamadas, identificar temas recurrentes |
| **Salud** | Dictar notas clínicas, clasificar por tema, generar resumen |

---

## ⚠️ Advertencias técnicas

- **Sesgo de Whisper**: entrenado ~70% en inglés. Español funciona bien, pero acentos muy marcados reducen precisión.
- **LDA con textos cortos**: con ~100 tokens, resultados varían entre ejecuciones. Segmentar en oraciones mitiga esto.
- **Groq free tier**: límite de ~14,400 solicitudes/día. Para producción usar plan de pago con SLA garantizado.

**Escalabilidad a producción:**
- Más de 100 horas de audio → `whisper-large-v3-turbo` con GPU A100 o API de OpenAI
- Corpus grande en LDA → `gensim.models.LdaMulticore` con paralelismo
- LLM en producción → Groq pago o despliegue propio con vLLM

## 📐 Matemática del pipeline (en lenguaje simple)

### ¿Cómo funciona Whisper? (ASR)

Whisper convierte audio en texto en **dos grandes pasos**:

**Paso 1 — El audio se convierte en "imagen" (Mel-spectrogram):**

Piénsalo como un piano fotográfico:
- El eje **Y** son las notas musicales (frecuencias, de grave a agudo)
- El eje **X** es el tiempo
- El **color** indica qué tan fuerte suena cada frecuencia en cada momento

```
Audio crudo:     ─────∿∿∿∿∿──────∿∿∿──────   ← números de amplitud
                            ↓  FFT cada 10ms
Mel-spectrogram: [imagen frecuencias × tiempo]  ← "foto del sonido"
```

**Paso 2 — Un Transformer traduce la imagen a texto (Encoder-Decoder):**

```
Mel-spectrogram ──► Encoder (CNN+Transformer) ──► Decoder (Transformer) ──► texto
                   "entiende" el audio           genera palabra por palabra
```

La fórmula central del Transformer es la **Atención Escalada**:

`Atención(Q, K, V) = softmax( QKᵀ / √dₖ ) · V`

**¿Qué significa en lenguaje simple?**
- `Q` (Query) = "¿qué información busco ahora?"
- `K` (Key) = "¿qué información está disponible?"
- `V` (Value) = la información en sí
- Resultado: cada palabra generada "mira" todas las partes del audio y decide cuáles son relevantes

---

### ¿Cómo funciona LDA? (Modelado de temas)

LDA asume que los documentos son **mezclas de temas**, y los temas son **distribuciones de palabras**:

`P(palabra w | documento d) = Σ_tópico P(w | tópico) × P(tópico | d)`

**En lenguaje simple:**

> La probabilidad de que aparezca "zorro" en una fábula = (prob. de "zorro" en el tema "animales") × (prob. de que esta fábula hable de "animales")

LDA aprende esto *al revés*: dado que observamos las palabras, ¿cuáles son los temas que mejor explican por qué aparecen juntas?

**¿Por qué segmentamos en oraciones?** LDA necesita ver co-ocurrencias *muchas veces*. Si "zorro" y "astuto" solo aparecen juntas en 2 de 100 tokens, LDA no aprende mucho. Si dividimos la fábula en 8 oraciones de ~12 tokens, LDA tiene 8 "documentos" con más co-ocurrencias visibles.

---

### ¿Cómo funciona el LLM? (Groq + Llama 3.3 70B)

Llama 3.3 genera texto **token por token**, donde cada token depende de todos los anteriores:

`P(texto) = P(t₁) × P(t₂|t₁) × P(t₃|t₁,t₂) × ... × P(tₙ|t₁...tₙ₋₁)`

**En lenguaje simple:**

> El modelo siempre responde la misma pregunta: "dado todo lo escrito antes, ¿cuál es la siguiente palabra más probable?"

Con 70 billones de parámetros entrenados en ~15 billones de tokens de texto humano, Llama "sabe" sobre literatura, fábulas de Esopo, y cómo resumir — sin entrenamiento adicional.

# **Ejercicio 1: Descarga de audios**

## ¿Cómo funciona la descarga HTTP? ¿Solo en Gutenberg?

`requests.get(url)` envía una solicitud **HTTP GET** estándar — el mismo mecanismo que usa tu navegador para cargar cualquier página o archivo:

```
Tu Python ──► HTTP GET https://sitio.com/archivo.mp3 ──► Servidor
             ◄── bytes del archivo MP3 ◄──
```

**¿Funciona en cualquier sitio?** Sí, con condiciones:

| Caso | ¿Funciona? | Ejemplo |
|------|-----------|--------|
| URL directa a MP3 público | ✅ Sí | Gutenberg, LibriVox, Wikipedia |
| Sitio con login requerido | ❌ No | Spotify, Audible |
| Contenido cargado por JavaScript | ❌ No | YouTube (necesitas `yt-dlp`) |
| API con token de autenticación | ✅ Sí | Añadir `headers={'Authorization': ...}` |

**Proyecto Gutenberg** es ideal porque:
- Todo el contenido es **dominio público** (sin derechos de autor)
- Las URLs de MP3 son **directas y predecibles**: `/files/ID/mp3/ID-NN.mp3`
- No requiere autenticación ni tiene rate limiting estricto

**Otras fuentes similares con descarga directa:**
- **LibriVox** (librivox.org) — audiolibros gratuitos leídos por voluntarios
- **Internet Archive** (archive.org) — millones de archivos de audio públicos
- **Wikipedia** — archivos de pronunciación en múltiples idiomas

In [ ]:
import os
import requests
from pathlib import Path

# Instalamos dependencias adicionales (Whisper ya está en ml_env)
!pip install groq gensim nltk --quiet

# ── Configuración ────────────────────────────────────────────────────────────
AUDIO_NUMBERS = [1, 4, 5, 6, 14, 22, 24, 25, 26, 27]   # fábulas solicitadas
AUDIO_DIR = Path('audios')      # carpeta local donde guardaremos los MP3
AUDIO_DIR.mkdir(exist_ok=True)  # exist_ok=True: no falla si ya existe

# {:02d} formatea el número con 2 dígitos: 1→'01', 14→'14'
BASE_URL = 'https://www.gutenberg.org/files/21144/mp3/21144-{:02d}.mp3'

print('Descargando audios — Proyecto Gutenberg (fábulas de Esopo)\n')

for num in AUDIO_NUMBERS:
    url = BASE_URL.format(num)
    filename = AUDIO_DIR / f'fabula_{num:02d}.mp3'

    # Si ya existe lo saltamos — permite re-ejecutar sin re-descargar
    if filename.exists():
        print(f'  Fábula {num:02d}: ya existe ({filename.stat().st_size/1024:.1f} KB)')
        continue

    print(f'  Descargando fábula {num:02d}... ', end='', flush=True)
    try:
        # stream=True: descarga en bloques (no carga todo en RAM a la vez)
        # timeout=60: cancela si el servidor no responde en 60 segundos
        r = requests.get(url, stream=True, timeout=60)
        r.raise_for_status()   # lanza error si responde 404, 403, 500, etc.

        with open(filename, 'wb') as f:   # 'wb' = write binary
            for chunk in r.iter_content(chunk_size=8192):  # bloques de 8 KB
                f.write(chunk)

        print(f'OK ({filename.stat().st_size/1024:.1f} KB)')
    except requests.exceptions.HTTPError as e:
        print(f'ERROR HTTP {e}')   # 404=no existe, 403=sin permiso
    except requests.exceptions.Timeout:
        print('ERROR: timeout')
    except Exception as e:
        print(f'ERROR: {e}')

downloaded = sorted(AUDIO_DIR.glob('*.mp3'))
print(f'\nTotal archivos MP3: {len(downloaded)}')
for f in downloaded:
    print(f'  {f.name}  —  {f.stat().st_size/1024:.1f} KB')

# **Ejercicio 2a: Transcripción de audio a texto (ASR)**

## ¿Qué hace `model.transcribe()` internamente?

```
fabula_01.mp3
    │
    ▼  Whisper carga el audio y calcula el Mel-spectrogram
[array de frecuencias × tiempo]
    │
    ▼  Encoder: convierte el audio en representación interna
[vector de 512 dimensiones por cada 20ms de audio]
    │
    ▼  Decoder: genera el texto token por token
'Las' → 'Las ranas' → 'Las ranas y' → ... → texto completo
    │
    ▼
result['text'] = 'Las ranas y el pantano seco...'
```

## Variantes de Whisper disponibles (todas gratuitas vía openai-whisper)

| Modelo | Tamaño | Tiempo en CPU / audio 1min | Recomendado para |
|--------|--------|---------------------------|------------------|
| `tiny` | 39 MB | ~30 seg | Prototipado rápido |
| `base` | 74 MB | ~1 min | Español con acento neutro |
| **`small`** ✅ | **244 MB** | **~3 min** | **Esta actividad** |
| `medium` | 769 MB | ~8 min | Mejor calidad sin GPU |
| `large-v3` | 1.5 GB | ~20 min | GPU obligatoria |

**`fp16=False`**: float16 usa la mitad de RAM y es 2x más rápido, pero solo en GPUs NVIDIA con CUDA. En Mac (CPU) usamos float32 para evitar errores silenciosos que producen texto vacío.

In [ ]:
import whisper
import json
from pathlib import Path

AUDIO_NUMBERS = [1, 4, 5, 6, 14, 22, 24, 25, 26, 27]
AUDIO_DIR = Path('audios')
RAW_JSON = Path('fabulas_raw.json')  # archivo de caché

# ── Recuperar caché si ya transcribimos antes ─────────────────────────────────
# Sin caché, cada vez que reiniciamos el kernel esperaríamos 20-40 min.
# Con caché, los ejercicios 3-5 se pueden correr independientemente.
if RAW_JSON.exists():
    with open(RAW_JSON, encoding='utf-8') as f:
        fabulas_raw = json.load(f)
    print(f'Transcripciones recuperadas desde caché ({len(fabulas_raw)} fábulas)\n')

else:
    # whisper.load_model() descarga los pesos la primera vez (~244 MB)
    # y los guarda en ~/.cache/whisper/ para siguientes ejecuciones
    print('Cargando Whisper-small...')
    model = whisper.load_model('small')

    fabulas_raw = {}  # {audioNN: texto_transcrito}
    print('\nTranscribiendo audios (~3 min/audio en CPU)\n')

    for num in AUDIO_NUMBERS:
        filename = AUDIO_DIR / f'fabula_{num:02d}.mp3'
        key = f'audio{num:02d}'  # 'audio01', 'audio04', ...

        print(f'  {key}: ', end='', flush=True)
        try:
            # language='es': fuerza español
            # Sin esto, Whisper detecta automáticamente pero puede fallar
            # en los primeros segundos con narradores de acento neutro
            #
            # fp16=False: obligatorio en CPU.
            # Cambiar a fp16=True solo si tienes GPU NVIDIA con CUDA.
            result = model.transcribe(str(filename), language='es', fp16=False)

            # result['text'] = transcripción completa como un string
            # .strip() elimina espacios al inicio/final
            fabulas_raw[key] = result['text'].strip()
            print(f'OK  ({len(result["text"])} chars)')

        except Exception as e:
            print(f'ERROR: {e}')

    # Guardar en JSON: ensure_ascii=False permite tildes y ñ
    with open(RAW_JSON, 'w', encoding='utf-8') as f:
        json.dump(fabulas_raw, f, ensure_ascii=False, indent=2)
    print(f'\nGuardado en {RAW_JSON}')

# Vista previa
print(f'\nResumen ({len(fabulas_raw)} fábulas):')
for key in sorted(fabulas_raw):
    print(f'\n  [{key}]  {fabulas_raw[key][:180]}...')

# **Ejercicio 2b: Eliminar intro y outro comunes**

## ¿Por qué existe un intro/outro?

LibriVox estandariza todas sus grabaciones:

```
[INTRO]  'Las fábulas de Esopo. Grabado para LibriVox.org por [nombre]. Fábula número N.'
[FÁBULA] 'Vivían dos ranas en un bello pantano...'
[OUTRO]  'Fin de la fábula. Esta es una grabación del dominio público. Gracias.'
```

El intro/outro añade palabras genéricas que confunden a LDA — aparecerían en todas las fábulas y no distinguen temas.

## ¿Cómo funcionan las expresiones regulares (regex)?

Son **patrones de búsqueda** para texto. Símbolos clave:

| Símbolo | Significado | Ejemplo |
|---------|-------------|--------|
| `.` | Cualquier carácter | `f.bula` → 'fabula' o 'fábula' |
| `*` | 0 o más veces | `espacios*` |
| `+` | 1 o más veces | `\\s+` = uno o más espacios |
| `?` | 0 o 1 vez (opcional) | `colou?r` → 'color' o 'colour' |
| `[aá]` | Uno de estos caracteres | 'a' o 'á' |
| `\\w` | Letra o dígito | |
| `\\s` | Espacio en blanco | |
| `^` | Inicio del texto | |
| `$` | Final del texto | |
| `[\\w\\s]+?` | Letras/dígitos/espacios, modo lazy (lo menos posible) | El número: '1', 'catorce', 'veintisiete' |

**¿Por qué `[\\w\\s]+?` para el número?** Whisper puede transcribir 'fábula número catorce' o 'fábula número 14'. El patrón flexible captura ambos.

In [ ]:
import re
import json

# ── Patrón de inicio ──────────────────────────────────────────────────────────
# Captura todo desde el principio hasta después del número de fábula
#
# ^         = desde el inicio del string
# .*?       = cualquier cosa, modo 'lazy' (lo MENOS posible — se detiene en el
#             primer match del resto del patrón, no en el último)
# f[aá]bula = 'fabula' o 'fábula'
# \s+       = uno o más espacios
# n[uú]mero = 'numero' o 'número'
# [\w\s]+? = el número (dígito como '14' o palabra como 'catorce')
# [.,]\s*  = punto o coma al final + espacios opcionales
#
# re.IGNORECASE: no distingue mayúsculas/minúsculas
# re.DOTALL:     el punto también captura saltos de línea
PATTERN_INICIO = re.compile(
    r'^.*?f[aá]bula\s+n[uú]mero\s+[\w\s]+?[.,]\s*',
    re.IGNORECASE | re.DOTALL
)

# ── Patrón de final ───────────────────────────────────────────────────────────
# Captura 'Fin de [la] fábula' y TODO lo que sigue
#
# \s*         = espacios opcionales antes
# [Ff]in      = 'Fin' o 'fin'
# (?:la\s+)? = 'la ' es opcional (algunos dicen 'Fin de fábula')
# [\s\S]*$  = todo lo que sigue hasta el final (incluye saltos de línea)
PATTERN_FIN = re.compile(
    r'\s*[Ff]in\s+de\s+(?:la\s+)?f[áa]bula[\s\S]*$',
    re.IGNORECASE
)

fabulas_clean = {}

print('Eliminando intro y outro comunes:\n')
print(f"  {'Fábula':<12} {'Antes':>8} {'Después':>9}  Reducción")
print('  ' + '-' * 38)

for key in sorted(fabulas_raw):
    texto = fabulas_raw[key]

    # .sub('', texto) reemplaza el patrón con string vacío (= borrarlo)
    # .strip() elimina espacios al inicio/final que puedan quedar
    texto_clean = PATTERN_INICIO.sub('', texto).strip()
    texto_clean = PATTERN_FIN.sub('', texto_clean).strip()

    fabulas_clean[key] = texto_clean
    print(f"  {key:<12} {len(texto):>8} {len(texto_clean):>9}  -{len(texto)-len(texto_clean)} chars")

# Guardar en JSON — punto de recuperación para ejercicios 3-5
with open('fabulas.json', 'w', encoding='utf-8') as f:
    json.dump(fabulas_clean, f, ensure_ascii=False, indent=2)

print('\nGuardado en fabulas.json')
for key in sorted(fabulas_clean):
    print(f'\n  [{key}]')
    print(f'  INICIO: {fabulas_clean[key][:100]}...')
    print(f'  FINAL:  ...{fabulas_clean[key][-70:]}')

# **Ejercicio 3: Limpieza de texto**

## ¿Por qué limpiar antes de LDA?

LDA trabaja con **frecuencias de palabras**. Sin limpieza, palabras como 'de', 'la', 'que' aparecen en todos los temas y los hacen idénticos.

Analogía: si analizas 100 recetas de cocina y las palabras más frecuentes son 'de', 'con', 'el' — no te dicen nada. En cambio 'pollo', 'harina', 'azúcar' sí distinguen temas.

## ¿Qué son las stopwords?

Palabras tan frecuentes que no aportan significado temático. NLTK incluye ~300 stopwords en español:

```
'de', 'la', 'que', 'el', 'en', 'y', 'a', 'los', 'del', 'se',
'las', 'por', 'un', 'para', 'con', 'una', 'su', 'al', 'lo', 'como'
```

## ¿Por qué NO aplicamos stemming?

El stemming reduce 'corriendo' → 'corr', 'corrió' → 'corr'. Con ~100 tokens por fábula, colapsar formas distintas puede eliminar distinciones semánticas relevantes. Con más texto (miles de documentos), el stemming sí ayuda.

In [ ]:
import re, json, nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Recuperar desde JSON si no está en memoria
if 'fabulas_clean' not in dir() or not fabulas_clean:
    with open('fabulas.json', encoding='utf-8') as f:
        fabulas_clean = json.load(f)

STOP_ES = set(stopwords.words('spanish'))  # ~300 palabras
MIN_LEN = 3  # filtrar residuos de 1-2 caracteres


def limpiar(texto: str) -> list:
    # Paso 1: minúsculas — 'León' y 'león' son la misma entidad
    texto = texto.lower()

    # Paso 2: eliminar puntuación
    # [^\w\s] = todo lo que NO sea letra, dígito o espacio
    # Reemplazamos con espacio para no unir palabras: 'fin,comenzó'→'fin comenzó'
    texto = re.sub(r'[^\w\s]', ' ', texto)

    # Paso 3: eliminar dígitos (residuos de transcripción)
    texto = re.sub(r'\d+', ' ', texto)

    # Paso 4: tokenizar con reglas de español
    tokens = word_tokenize(texto, language='spanish')

    # Paso 5: filtrar — solo letras, longitud mínima, sin stopwords
    return [t for t in tokens
            if t.isalpha() and len(t) >= MIN_LEN and t not in STOP_ES]


fabulas_tokens = {k: limpiar(v) for k, v in fabulas_clean.items()}

print(f"  {'Fábula':<12} {'Palabras orig':>14} {'Tokens limpios':>15}  Retención")
print('  ' + '-' * 52)
for key in sorted(fabulas_tokens):
    orig = len(fabulas_clean[key].split())
    tok = len(fabulas_tokens[key])
    print(f"  {key:<12} {orig:>14} {tok:>15}  {tok/orig*100:.0f}%")

print('\nMuestra (primeros 15 tokens):')
for key in sorted(fabulas_tokens):
    print(f'  {key}: {fabulas_tokens[key][:15]}')

# **Ejercicio 4: LDA — Modelado de temas**

## ¿Qué hace LDA exactamente?

LDA aprende qué palabras tienden a aparecer juntas y las agrupa en 'temas'. Para esta actividad pedimos **1 tópico con 20 palabras** por fábula.

**Fórmula central:**
```
P(palabra w | documento d) = Σ_tópico P(w | tópico) × P(tópico | d)
```

**En lenguaje simple:** la probabilidad de ver 'zorro' en una fábula = (prob. de 'zorro' en el tema 'animales astutos') × (prob. de que esta fábula hable de 'animales astutos')

## El problema del corpus corto y la solución

```
Sin segmentar:                    Segmentando en oraciones:
  fabula_01 (1 doc, 100 tokens)     Oración 1: 'Vivían dos ranas...'   (12 tokens)
                                    Oración 2: 'Pero llegó el verano...' (10 tokens)
  → LDA ve pocas co-ocurrencias     Oración 3: 'Aliaron en su camino...' (14 tokens)
  → Resultados inestables           ...
                                    → 6-8 documentos por fábula
                                    → más co-ocurrencias visibles
                                    → LDA más estable
```

## Parámetros clave de `gensim.LdaModel`

| Parámetro | Valor | Qué controla |
|-----------|-------|-------------|
| `num_topics=1` | 1 | 1 solo tema por fábula |
| `passes=50` | 50 | Iteraciones sobre el corpus — más = más estable |
| `random_state=42` | 42 | Semilla aleatoria para reproducibilidad |
| `minimum_probability=0.0` | 0.0 | Incluir palabras con cualquier probabilidad |

In [ ]:
import re
from collections import Counter
from gensim import corpora, models

NUM_WORDS = 20
fabulas_keywords = {}  # resultado: {audioNN: [lista 20 palabras]}

print(f'=== LDA — 1 tópico, {NUM_WORDS} palabras por fábula ===\n')

for key in sorted(fabulas_clean):
    texto = fabulas_clean[key]

    # Paso 1: segmentar en oraciones
    # re.split(r'[.!?]+') divide usando . ! ? como separadores
    # El '+' en [.!?]+ maneja '...' sin crear segmentos vacíos
    oraciones = [s.strip() for s in re.split(r'[.!?]+', texto) if s.strip()]

    # Paso 2: limpiar cada oración con la función del Ej 3
    # Filtramos oraciones con <2 tokens (demasiado cortas para LDA)
    sent_tokens = [limpiar(s) for s in oraciones]
    sent_tokens = [t for t in sent_tokens if len(t) >= 2]

    # Fallback 1: pocas oraciones → usar texto completo como 1 documento
    if len(sent_tokens) < 3:
        sent_tokens = [fabulas_tokens[key]]

    # Fallback 2: sin tokens → usar frecuencia simple (no LDA)
    if not sent_tokens or not any(sent_tokens):
        counter = Counter(fabulas_tokens[key])
        fabulas_keywords[key] = [w for w, _ in counter.most_common(NUM_WORDS)]
        print(f'  {key} (fallback frecuencia): {chr(44)+chr(32).join(fabulas_keywords[key])}\n')
        continue

    # Paso 3: construir vocabulario
    # Dictionary mapea cada palabra única a un ID numérico
    # Ejemplo: {'ranas': 0, 'pantano': 1, 'agua': 2, ...}
    fable_dict = corpora.Dictionary(sent_tokens)

    # Paso 4: convertir a Bag-of-Words
    # doc2bow(['ranas', 'pantano', 'ranas']) → [(0,2), (1,1)]
    # Cada tupla es (id_palabra, frecuencia_en_este_documento)
    fable_corpus = [fable_dict.doc2bow(doc) for doc in sent_tokens]

    # Paso 5: entrenar LDA
    lda = models.LdaModel(
        corpus=fable_corpus,
        id2word=fable_dict,
        num_topics=1,          # 1 tópico por fábula
        passes=50,             # 50 iteraciones para estabilidad
        random_state=42,       # reproducibilidad
        minimum_probability=0.0
    )

    # Paso 6: extraer las 20 palabras del único tópico
    # show_topic(0, topn=20) devuelve [(palabra, probabilidad), ...]
    # Nos quedamos solo con las palabras (no mostramos las probabilidades)
    fabulas_keywords[key] = [w for w, _ in lda.show_topic(0, topn=NUM_WORDS)]

    n_docs = len(sent_tokens)
    n_tok = sum(len(t) for t in sent_tokens)
    print(f'  {key} ({n_docs} oraciones / {n_tok} tokens):')
    print(f'    {", ".join(fabulas_keywords[key])}\n')

print(f'Palabras clave extraídas para {len(fabulas_keywords)} fábulas.')

# **Ejercicio 5a y 5b: Generación con LLM (Groq)**

## ¿Qué es Groq y por qué es tan rápido?

> **Groq ≠ Grok**. Groq es una empresa independiente (ex-ingenieros de Google TPU) con su propio chip **LPU**. Grok es el modelo de xAI (Elon Musk). Son empresas completamente distintas.

El **LPU** está optimizado para la naturaleza secuencial de los LLMs:
```
GPU (paralelo masivo):  ideal para imágenes, matrices enormes
LPU (secuencial rápido): ideal para generar un token a la vez → ~500 tok/seg
```

## ¿Qué es el prompt engineering?

Un buen prompt tiene 4 partes:
1. **Rol** — define el tono y conocimiento esperado del modelo
2. **Contexto** — los datos de entrada (palabras clave, texto)
3. **Tarea explícita** — qué debe producir
4. **Formato fijo** — estructura para parsear la respuesta automáticamente

**¿Por qué formato fijo?** Si el LLM responde en texto libre, extraer el resumen y subtemas de 10 fábulas automáticamente es difícil. Con etiquetas fijas (`RESUMEN:`, `SUBTEMA_N:`), el parseo es una simple búsqueda de prefijos.

## Parámetro `temperature`

| Valor | Comportamiento | Cuándo usarlo |
|-------|---------------|---------------|
| `0.0` | Determinístico — siempre la misma respuesta | Extracción de datos, clasificación |
| `0.7` ✅ | Balance creatividad/coherencia | Resúmenes, descripciones |
| `1.0+` | Muy creativo, puede ser incoherente | Poesía, ficción |

In [ ]:
import os, json, getpass
from groq import Groq

api_key = os.environ.get('GROQ_API_KEY') or getpass.getpass('GROQ_API_KEY: ')
client = Groq(api_key=api_key)
MODEL = 'llama-3.3-70b-versatile'

def analizar_fabula(keywords: list, texto: str) -> dict:
    # El prompt tiene 4 partes: rol, datos, tarea y formato fijo para parseo
    prompt = (
        'Eres un experto en literatura clasica y fabulas de Esopo.\n\n'
        f'Palabras clave (LDA): {", ".join(keywords)}\n'
        f'Texto de la fabula: {texto[:500]}\n\n'
        'Responde UNICAMENTE en este formato exacto (sin texto adicional):\n'
        'RESUMEN: [enunciado de max 30 palabras que describa la fabula]\n'
        'SUBTEMA_1: [situacion especifica de la historia, max 25 palabras]\n'
        'SUBTEMA_2: [otra situacion diferente, max 25 palabras]\n'
        'SUBTEMA_3: [tercera situacion diferente, max 25 palabras]\n\n'
        'Responde en espanol.'
    )
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.7,
        max_tokens=400
    )
    raw = response.choices[0].message.content
    result = {'resumen': '', 'subtemas': [], 'raw': raw}
    for line in raw.strip().split('\n'):
        line = line.strip()
        if line.startswith('RESUMEN:'):
            result['resumen'] = line[8:].strip()
        elif line.startswith('SUBTEMA_'):
            result['subtemas'].append(':'.join(line.split(':')[1:]).strip())
    return result

print(f'=== Ejercicio 5 - Groq / {MODEL} ===\n')
fabulas_llm = {}

for key in sorted(fabulas_keywords):
    print(f'Procesando {key}...', flush=True)
    res = analizar_fabula(fabulas_keywords[key], fabulas_clean.get(key, ''))
    fabulas_llm[key] = res
    print(f'  5a) {res["resumen"]}')
    for i, s in enumerate(res['subtemas'], 1):
        print(f'  5b.{i}) {s}')
    print()

with open('fabulas_llm.json', 'w', encoding='utf-8') as f:
    json.dump(fabulas_llm, f, ensure_ascii=False, indent=2)
print('Guardado en fabulas_llm.json')

# **Ejercicio 6: Conclusiones**

## Conclusiones de la actividad audio-a-texto

### 1. Transcripción (Whisper-small local)
Whisper-small produjo transcripciones de alta calidad en los 10 audios. `language='es'` fue crítico para evitar confusión de idioma. El caché en `fabulas_raw.json` evita re-transcribir (~3 min/audio en CPU) al reiniciar el kernel.

### 2. Eliminación de intro/outro
Whisper transcribe números inconsistentemente ('catorce' vs '14'). El patrón `[\\w\\s]+?` en modo lazy resuelve ambos casos sin ajuste manual por fábula.

### 3. Limpieza de texto
La estrategia conservadora fue correcta: con ~100-150 tokens por fábula, aplicar stemming o umbrales de frecuencia habría dejado documentos demasiado escasos para LDA.

### 4. LDA — Modelado de temas
LDA necesita corpus grandes para ser estable. La segmentación en oraciones (8 documentos × 12 tokens vs 1 documento × 100 tokens) mejoró la coherencia. Para corpus cortos en producción: **KeyBERT** (embeddings BERT) o **YAKE** (sin entrenamiento).

### 5. LLM — Groq con Llama 3.3 70B
El LLM produjo resúmenes coherentes incluso con keywords de LDA subóptimas, al usar el texto completo como contexto adicional. El formato fijo en el prompt hizo el parseo automático robusto para las 10 fábulas.

### Tabla resumen del stack

| Etapa | Herramienta | Costo | Privacidad |
|-------|------------|-------|------------|
| Descarga | `requests` | $0 | N/A (datos públicos) |
| ASR | Whisper-small local | $0 | Total — audio no sale de la máquina |
| Limpieza | NLTK | $0 | Total |
| Temas | gensim LDA | $0 | Total |
| LLM | Groq free tier | $0 | Texto enviado a Groq |

**Lección clave:** la calidad de cada etapa condiciona la siguiente. Verificar la transcripción (Ej 2a) antes de continuar es la decisión más importante del pipeline.

# **Fin de la guía de estudio — Semanas 7 y 8**